# CAD/USD traditional forecasting baselines

This notebook compares three baseline families before adding an agent:

- **AutoARIMA**: univariate statistical benchmark using CAD/USD history only.
- **LinearRegression**: transparent lagged-feature model.
- **LightGBM**: nonlinear tree model that can learn interactions.

The comparison uses the same targets, forecast origins, and evaluation metric for every model. `RUN_BACKTEST` is `False` by default because AutoARIMA can take time.

In [6]:
from pathlib import Path
import sys

import pandas as pd
import yaml
from IPython.display import display

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'aieng-forecasting').is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from aieng.forecasting.evaluation import MultiTargetBacktestSpec, cached_multi_backtest
from aieng.forecasting.methods import (
    DartsAutoARIMAPredictor,
    DartsLightGBMPredictor,
    DartsLinearRegressionPredictor,
)
from implementations.cad_usd_forecasting.data import build_cadusd_service

RUN_BACKTEST = True
RUN_AUTOARIMA = False
SPEC_PATH = ROOT / 'implementations' / 'cad_usd_forecasting' / 'specs' / 'cadusd_smoke.yaml'
PREDICTIONS_DIR = ROOT / 'data' / 'predictions' / 'cadusd'

print('RUN_BACKTEST =', RUN_BACKTEST)
print('RUN_AUTOARIMA =', RUN_AUTOARIMA)
print('Spec:', SPEC_PATH)

RUN_BACKTEST = True
RUN_AUTOARIMA = False
Spec: /home/coder/development/agentic-forecasting/implementations/cad_usd_forecasting/specs/cadusd_smoke.yaml


## Why these three models?

AutoARIMA is the history-only benchmark. LinearRegression tells us whether lagged relationships are useful in a simple, interpretable form. LightGBM tests whether nonlinear effects and interactions improve the forecast.

We run LinearRegression and LightGBM twice: once with CAD/USD history only and once with the lagged WTI covariate. This isolates the value of the covariate from the value of the model family.

In [7]:
with SPEC_PATH.open(encoding='utf-8') as handle:
    spec = MultiTargetBacktestSpec.model_validate(yaml.safe_load(handle))

service = build_cadusd_service(
    start='1990-01-01',
    include_covariates=True,
)
covariates = ['wti_log_ret_1b_l1b']
registered = set(service.series_ids)
covariates = [series_id for series_id in covariates if series_id in registered]

print(spec.description)
print('Tasks:', [task.task_id for task in spec.tasks])
print('Covariates available:', covariates)

Smoke comparison of AutoARIMA, linear regression, and LightGBM for CAD/USD cumulative log returns at 1, 5, and 21 business-day horizons.
Tasks: ['cadusd_logret_1b', 'cadusd_logret_5b', 'cadusd_logret_21b']
Covariates available: ['wti_log_ret_1b_l1b']


In [8]:
LAGS = 21
NUM_SAMPLES = 100

predictors = {
    'LinearRegression': DartsLinearRegressionPredictor(
        lags=LAGS,
        covariate_series_ids=None,
        num_samples=NUM_SAMPLES,
    ),
    'LinearRegression + WTI': DartsLinearRegressionPredictor(
        lags=LAGS,
        lags_past_covariates=LAGS,
        covariate_series_ids=covariates,
        num_samples=NUM_SAMPLES,
    ),
    'LightGBM': DartsLightGBMPredictor(
        lags=LAGS,
        covariate_series_ids=None,
        num_samples=NUM_SAMPLES,
    ),
    'LightGBM + WTI': DartsLightGBMPredictor(
        lags=LAGS,
        lags_past_covariates=LAGS,
        covariate_series_ids=covariates,
        num_samples=NUM_SAMPLES,
    ),
}
if RUN_AUTOARIMA:
    predictors = {
        'AutoARIMA': DartsAutoARIMAPredictor(num_samples=NUM_SAMPLES),
        **predictors,
    }

for name, predictor in predictors.items():
    print(name, '->', predictor.predictor_id)

LinearRegression -> darts_linreg
LinearRegression + WTI -> darts_linreg_cov
LightGBM -> darts_lightgbm
LightGBM + WTI -> darts_lightgbm_cov


## Run the comparison

The default smoke run excludes AutoARIMA because it refits a model at every forecast origin and can be very slow. Set `RUN_AUTOARIMA = True` only when you want that separate benchmark. Each origin trains the predictor only on observations available before that origin.

CRPS is the main metric: lower is better. It evaluates the full probabilistic forecast, not only the median.

In [ ]:
results = {}
if RUN_BACKTEST:
    for name, predictor in predictors.items():
        print(f'Running {name} ...', flush=True)
        results[name] = cached_multi_backtest(
            predictor=predictor,
            spec=spec,
            data_service=service,
            store_dir=PREDICTIONS_DIR,
        )
else:
    print('Backtest disabled. Set RUN_BACKTEST = True to run it.')

Running LinearRegression ...


In [ ]:
rows = []
for name, task_results in results.items():
    for task_id, result in task_results.items():
        rows.append({
            'model': name,
            'task': task_id,
            'mean_crps': result.mean_score,
            'predictions': len(result.predictions),
            'scores': len(result.scores),
        })

results_df = pd.DataFrame(rows)
if results_df.empty:
    print('No results yet. Enable and run the backtest cell first.')
else:
    display(results_df.sort_values(['task', 'mean_crps']))

## How to interpret the first result

- If AutoARIMA wins, recent CAD/USD history may contain most of the useful signal.
- If `LinearRegression + WTI` improves over LinearRegression, WTI adds a linear relationship.
- If `LightGBM + WTI` wins, nonlinear effects or interactions may matter.
- If WTI variants do not improve, that is still useful evidence: this covariate may not help at these horizons.

Do not choose a model from one smoke window. Use this notebook to verify the pipeline, then expand the backtest window and reserve a later period for protected evaluation.